# Predictable-Drift Diagnostic for FTS-Diffusion

This notebook is a Colab-ready version of the `drift_component` code.

**Central point:** FTS-Diffusion can match the unconditional appearance of financial returns, but that does not imply a no-predictable-drift condition such as

$$\mathbb{E}[r_{t+1}\mid \mathcal{F}_t] = 0.$$

We therefore compute the fixed diagnostic suggested in the supervisor note:

$$\omega_t = (1, r_t, r_{t-1}, |r_t|, |r_{t-1}|),$$

$$\widehat{\delta} = \left\|\frac{1}{T}\sum_t r_{t+1}\omega_t\right\|_2.$$

Run this notebook on real returns/prices and one or more generated FTS-Diffusion sample CSVs. It also creates a block-shuffled null benchmark and stylized-fact summaries.

**Important caveat:** lower `delta` means less predictable drift under this fixed finite-dimensional feature map. It is not a full martingale or no-arbitrage proof.

## 1. Configure Inputs

In Colab, either upload files with the upload cell below, mount Google Drive, or set paths manually. Expected synthetic files should be ordered generated returns unless you explicitly set `SYNTHETIC_INPUT_TYPE` otherwise.

Valid input types:

- `return`: selected numeric column is already simple returns
- `value`: selected numeric column is already the target value series and is treated like returns for the diagnostic
- `close`: selected numeric column is prices; simple returns are computed with `pct_change()`
- `log_return`: selected numeric column is prices; log returns are computed

In [ ]:
# Optional Colab upload helper. Run this cell if your CSVs are on your computer.
# After upload, set REAL_CSV and SYNTHETIC_CSVS to the uploaded filenames.
try:
    from google.colab import files
    uploaded = files.upload()
    print('Uploaded:', list(uploaded.keys()))
except Exception as exc:
    print('Upload helper is only available in Colab:', exc)

In [ ]:
from pathlib import Path

# ---- EDIT THESE PATHS ----
# Example for uploaded files:
# REAL_CSV = Path('sp500_real.csv')
# SYNTHETIC_CSVS = [Path('generated_seed_0.csv'), Path('generated_seed_1.csv')]

REAL_CSV = None
SYNTHETIC_CSVS = []

# If REAL_CSV is prices, use INPUT_TYPE = 'close'.
# If REAL_CSV is returns, use INPUT_TYPE = 'return'.
INPUT_TYPE = 'return'

# Generated FTS-Diffusion files are usually returns.
SYNTHETIC_INPUT_TYPE = 'return'

# Optional column names. Leave None to infer from common names.
VALUE_COLUMN = None
DATE_COLUMN = None

# Chronological split for training statistics and evaluation.
TRAIN_FRACTION = 0.8
VAL_FRACTION = 0.0
TEST_FRACTION = None
EVAL_SPLIT = 'test'  # train, val, test, or full

# Diagnostic settings.
WINDOW_SIZE = 252
STEP_SIZE = 21
MIN_WINDOW = 64
BLOCK_SIZE = 21
NULL_REPS = 20
SEED = 0
MAX_LAG = 50

OUTPUT_DIR = Path('drift_diagnostics_colab')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Configured output dir:', OUTPUT_DIR.resolve())

## 2. Imports and Core Functions

In [ ]:
import json
import math
from typing import Any

import numpy as np
import pandas as pd

DRIFT_COMPONENTS = (
    'mean_r_next',
    'mean_r_next_r_t',
    'mean_r_next_r_t_minus_1',
    'mean_r_next_abs_r_t',
    'mean_r_next_abs_r_t_minus_1',
)

COMMON_COLUMNS = ('return', 'returns', 'r', 'close', 'value', 'generated', 'synthetic')


def as_1d_float_array(x: Any) -> np.ndarray:
    if isinstance(x, pd.DataFrame):
        numeric = x.select_dtypes(include=['number'])
        if numeric.shape[1] != 1:
            raise ValueError('DataFrame input must have exactly one numeric column.')
        values = numeric.iloc[:, 0].to_numpy(dtype=float)
    elif isinstance(x, (pd.Series, pd.Index)):
        values = x.to_numpy(dtype=float)
    else:
        values = np.asarray(x, dtype=float)
    values = np.ravel(values).astype(float, copy=False)
    return values[np.isfinite(values)]


def train_standardize(series: Any, train_mean: float | None, train_std: float | None, eps: float = 1e-8) -> np.ndarray:
    values = as_1d_float_array(series)
    mean = float(np.mean(values)) if train_mean is None else float(train_mean)
    std = float(np.std(values, ddof=0)) if train_std is None else float(train_std)
    denom = std if abs(std) > eps else eps
    return (values - mean) / denom


def make_omega(returns: Any, standardize: bool = True, train_mean: float | None = None, train_std: float | None = None, eps: float = 1e-8) -> np.ndarray:
    values = train_standardize(returns, train_mean, train_std, eps=eps) if standardize else as_1d_float_array(returns)
    if values.size < 3:
        return np.empty((0, 5), dtype=float)
    r_t = values[1:-1]
    r_t_minus_1 = values[:-2]
    return np.column_stack([np.ones_like(r_t), r_t, r_t_minus_1, np.abs(r_t), np.abs(r_t_minus_1)])


def drift_moment(returns: Any, standardize: bool = True, train_mean: float | None = None, train_std: float | None = None, eps: float = 1e-8) -> np.ndarray:
    values = train_standardize(returns, train_mean, train_std, eps=eps) if standardize else as_1d_float_array(returns)
    if values.size < 3:
        return np.full(5, np.nan, dtype=float)
    omega = make_omega(values, standardize=False)
    r_next = values[2:]
    return np.mean(r_next[:, None] * omega, axis=0)


def drift_delta(returns: Any, standardize: bool = True, train_mean: float | None = None, train_std: float | None = None, eps: float = 1e-8) -> float:
    moment = drift_moment(returns, standardize=standardize, train_mean=train_mean, train_std=train_std, eps=eps)
    return float(np.linalg.norm(moment)) if np.all(np.isfinite(moment)) else float('nan')


def drift_report(returns: Any, standardize: bool = True, train_mean: float | None = None, train_std: float | None = None, eps: float = 1e-8) -> dict[str, Any]:
    values = as_1d_float_array(returns)
    mean = float(np.mean(values)) if standardize and train_mean is None else train_mean
    std = float(np.std(values, ddof=0)) if standardize and train_std is None else train_std
    moment = drift_moment(values, standardize=standardize, train_mean=mean, train_std=std, eps=eps)
    report = {
        'n_obs': int(values.size),
        'delta': float(np.linalg.norm(moment)) if np.all(np.isfinite(moment)) else float('nan'),
        'train_mean': mean,
        'train_std': std,
    }
    report.update({name: float(value) for name, value in zip(DRIFT_COMPONENTS, moment)})
    return report


def rolling_drift_report(returns: Any, window_size: int = 252, step_size: int = 21, min_window: int = 64, standardize: bool = True, train_mean: float | None = None, train_std: float | None = None, eps: float = 1e-8) -> dict[str, Any]:
    values = as_1d_float_array(returns)
    pooled = drift_report(values, standardize=standardize, train_mean=train_mean, train_std=train_std, eps=eps)
    if values.size >= window_size:
        slices = [(start, start + window_size) for start in range(0, values.size - window_size + 1, step_size)]
    elif values.size >= min_window:
        slices = [(0, values.size)]
    else:
        slices = []
    windows = []
    for start, end in slices:
        row = drift_report(values[start:end], standardize=standardize, train_mean=train_mean, train_std=train_std, eps=eps)
        row['start'] = int(start)
        row['end'] = int(end)
        windows.append(row)
    deltas = np.asarray([row['delta'] for row in windows], dtype=float)
    finite = deltas[np.isfinite(deltas)]
    return {
        'pooled_delta': float(pooled['delta']),
        'rolling_delta_mean': float(np.mean(finite)) if finite.size else float('nan'),
        'rolling_delta_std': float(np.std(finite, ddof=0)) if finite.size else float('nan'),
        'rolling_delta_min': float(np.min(finite)) if finite.size else float('nan'),
        'rolling_delta_max': float(np.max(finite)) if finite.size else float('nan'),
        'n_windows': int(len(windows)),
        'windows': windows,
        'pooled_report': pooled,
    }


def block_shuffle_returns(returns: Any, block_size: int = 21, seed: int | None = None) -> np.ndarray:
    values = as_1d_float_array(returns)
    blocks = [values[start:start + block_size] for start in range(0, values.size, block_size)]
    if not blocks:
        return values.copy()
    order = np.random.default_rng(seed).permutation(len(blocks))
    return np.concatenate([blocks[index] for index in order])


def null_drift_reports(returns: Any, n_reps: int = 20, block_size: int = 21, seed: int = 0, train_mean: float | None = None, train_std: float | None = None) -> list[dict[str, Any]]:
    rng = np.random.default_rng(seed)
    rows = []
    for rep in range(n_reps):
        shuffled = block_shuffle_returns(returns, block_size=block_size, seed=int(rng.integers(0, 2**32 - 1)))
        row = drift_report(shuffled, train_mean=train_mean, train_std=train_std)
        row['rep'] = rep
        rows.append(row)
    return rows

print('Core drift functions loaded.')

In [ ]:
def summary_stats(returns: Any) -> dict[str, Any]:
    values = as_1d_float_array(returns)
    if values.size == 0:
        return {'n_obs': 0, 'mean': np.nan, 'std': np.nan, 'skew': np.nan, 'excess_kurtosis': np.nan, 'min': np.nan, 'max': np.nan, 'q01': np.nan, 'q05': np.nan, 'q50': np.nan, 'q95': np.nan, 'q99': np.nan}
    mean = float(np.mean(values))
    std = float(np.std(values, ddof=0))
    z = (values - mean) / std if std > 0 else np.full_like(values, np.nan)
    q = np.quantile(values, [0.01, 0.05, 0.50, 0.95, 0.99])
    return {
        'n_obs': int(values.size),
        'mean': mean,
        'std': std,
        'skew': float(np.nanmean(z**3)),
        'excess_kurtosis': float(np.nanmean(z**4) - 3.0),
        'min': float(np.min(values)),
        'max': float(np.max(values)),
        'q01': float(q[0]),
        'q05': float(q[1]),
        'q50': float(q[2]),
        'q95': float(q[3]),
        'q99': float(q[4]),
    }


def acf(x: Any, max_lag: int = 50) -> np.ndarray:
    values = as_1d_float_array(x)
    result = np.full(max_lag + 1, np.nan, dtype=float)
    if values.size == 0:
        return result
    centered = values - np.mean(values)
    denom = float(np.dot(centered, centered))
    if denom <= 0:
        result[0] = 1.0
        return result
    result[0] = 1.0
    for lag in range(1, min(max_lag, values.size - 1) + 1):
        result[lag] = float(np.dot(centered[:-lag], centered[lag:]) / denom)
    return result


def stylized_report(returns: Any, max_lag: int = 50) -> dict[str, Any]:
    values = as_1d_float_array(returns)
    stats = summary_stats(values)
    acf_r = acf(values, max_lag=max_lag)
    acf_abs = acf(np.abs(values), max_lag=max_lag)
    upper = min(max_lag, max(values.size - 1, 0))
    return {
        **stats,
        'acf_r_lag1': float(acf_r[1]) if max_lag >= 1 and acf_r.size > 1 else float('nan'),
        'acf_abs_r_lag1': float(acf_abs[1]) if max_lag >= 1 and acf_abs.size > 1 else float('nan'),
        'acf_r_l1_mean_abs': float(np.nanmean(np.abs(acf_r[1:upper + 1]))) if upper >= 1 else float('nan'),
        'acf_abs_r_l1_mean_abs': float(np.nanmean(np.abs(acf_abs[1:upper + 1]))) if upper >= 1 else float('nan'),
    }

print('Stylized-fact functions loaded.')

## 3. CSV Loading and Splitting

In [ ]:
def infer_value_column(df: pd.DataFrame, requested: str | None = None) -> str:
    if requested:
        if requested not in df.columns:
            raise ValueError(f'Column {requested!r} not found. Available columns: {list(df.columns)}')
        return requested
    lower_to_original = {str(column).lower(): column for column in df.columns}
    for candidate in COMMON_COLUMNS:
        if candidate in lower_to_original:
            return str(lower_to_original[candidate])
    numeric = df.select_dtypes(include=['number'])
    if numeric.shape[1] >= 1:
        return str(numeric.columns[0])
    raise ValueError('Could not infer a numeric value column.')


def load_return_series(path: Path, value_column: str | None = None, date_column: str | None = None, input_type: str = 'return') -> np.ndarray:
    df = pd.read_csv(path)
    if date_column and date_column in df.columns:
        df = df.sort_values(date_column)
    column = infer_value_column(df, value_column)
    values = pd.to_numeric(df[column], errors='coerce').dropna()
    if input_type in {'return', 'value'}:
        returns = values
    elif input_type == 'close':
        returns = values.pct_change().dropna()
    elif input_type == 'log_return':
        returns = np.log(values / values.shift(1)).dropna()
    else:
        raise ValueError(f'Unsupported input_type: {input_type}')
    return as_1d_float_array(returns)


def chronological_split(values: np.ndarray, train_fraction: float, val_fraction: float, test_fraction: float | None) -> dict[str, np.ndarray]:
    n = values.size
    train_end = int(math.floor(n * train_fraction))
    val_end = train_end + int(math.floor(n * val_fraction))
    test_end = val_end + int(math.floor(n * test_fraction)) if test_fraction is not None else n
    return {'train': values[:train_end], 'val': values[train_end:val_end], 'test': values[val_end:test_end], 'full': values}

print('CSV loader loaded.')

## 4. Run Evaluation

If `REAL_CSV` or `SYNTHETIC_CSVS` are not set, this cell runs a toy smoke experiment. That is useful for checking that the notebook works, but it is not an FTS-Diffusion result.

In [ ]:
def write_toy_data(output_dir: Path, seed: int = 0, length: int = 2048) -> tuple[Path, list[Path]]:
    rng = np.random.default_rng(seed)
    iid = rng.normal(size=length)
    noise = rng.normal(size=length)
    ar = np.empty(length)
    ar[0] = noise[0]
    for i in range(1, length):
        ar[i] = 0.4 * ar[i - 1] + noise[i]
    null = block_shuffle_returns(iid, block_size=BLOCK_SIZE, seed=seed + 100)
    toy_dir = output_dir / 'toy_inputs'
    toy_dir.mkdir(parents=True, exist_ok=True)
    real_path = toy_dir / 'iid_gaussian_returns.csv'
    ar_path = toy_dir / 'ar1_phi_0_4_returns.csv'
    null_path = toy_dir / 'block_shuffled_iid_returns.csv'
    pd.DataFrame({'return': iid}).to_csv(real_path, index=False)
    pd.DataFrame({'return': ar}).to_csv(ar_path, index=False)
    pd.DataFrame({'return': null}).to_csv(null_path, index=False)
    return real_path, [ar_path, null_path]


if REAL_CSV is None or not SYNTHETIC_CSVS:
    print('No real/synthetic CSV paths configured. Running toy smoke experiment.')
    REAL_CSV, SYNTHETIC_CSVS = write_toy_data(OUTPUT_DIR, seed=SEED)
    INPUT_TYPE = 'return'
    SYNTHETIC_INPUT_TYPE = 'return'
    VALUE_COLUMN = 'return'

real_returns = load_return_series(Path(REAL_CSV), value_column=VALUE_COLUMN, date_column=DATE_COLUMN, input_type=INPUT_TYPE)
splits = chronological_split(real_returns, TRAIN_FRACTION, VAL_FRACTION, TEST_FRACTION)
train = splits['train']
real_eval = splits[EVAL_SPLIT]
if train.size == 0:
    raise ValueError('Training split is empty.')
if real_eval.size < 3:
    raise ValueError(f'Eval split {EVAL_SPLIT!r} has fewer than 3 observations.')

train_mean = float(np.mean(train))
train_std = float(np.std(train, ddof=0))

print('Real CSV:', REAL_CSV)
print('Synthetic CSVs:', SYNTHETIC_CSVS)
print('Train observations:', len(train), 'Eval observations:', len(real_eval))
print('Train mean/std:', train_mean, train_std)

In [ ]:
def source_id(path: Path) -> str:
    return Path(path).stem


def row_triplet(source_type: str, sample_id: str, returns: np.ndarray) -> tuple[dict[str, Any], dict[str, Any], dict[str, Any]]:
    pooled = drift_report(returns, train_mean=train_mean, train_std=train_std)
    rolling = rolling_drift_report(returns, window_size=WINDOW_SIZE, step_size=STEP_SIZE, min_window=MIN_WINDOW, train_mean=train_mean, train_std=train_std)
    stylized = stylized_report(returns, max_lag=MAX_LAG)
    drift_row = {
        'source_type': source_type,
        'sample_id': sample_id,
        'n_obs': pooled['n_obs'],
        'pooled_delta': pooled['delta'],
        'rolling_delta_mean': rolling['rolling_delta_mean'],
        'rolling_delta_std': rolling['rolling_delta_std'],
        'rolling_delta_min': rolling['rolling_delta_min'],
        'rolling_delta_max': rolling['rolling_delta_max'],
        'n_windows': rolling['n_windows'],
        'train_mean': pooled['train_mean'],
        'train_std': pooled['train_std'],
    }
    component_row = {'source_type': source_type, 'sample_id': sample_id, **{name: pooled[name] for name in DRIFT_COMPONENTS}}
    stylized_row = {'source_type': source_type, 'sample_id': sample_id, **stylized}
    return drift_row, component_row, stylized_row


drift_rows = []
component_rows = []
stylized_rows = []

for target, row in zip((drift_rows, component_rows, stylized_rows), row_triplet('real', EVAL_SPLIT, real_eval)):
    target.append(row)

for path in SYNTHETIC_CSVS:
    synthetic_returns = load_return_series(Path(path), value_column=VALUE_COLUMN, date_column=DATE_COLUMN, input_type=SYNTHETIC_INPUT_TYPE)
    for target, row in zip((drift_rows, component_rows, stylized_rows), row_triplet('synthetic', source_id(Path(path)), synthetic_returns)):
        target.append(row)

rng = np.random.default_rng(SEED)
null_rows = []
null_stylized_rows = []
for rep in range(NULL_REPS):
    shuffled = block_shuffle_returns(real_eval, block_size=BLOCK_SIZE, seed=int(rng.integers(0, 2**32 - 1)))
    pooled = drift_report(shuffled, train_mean=train_mean, train_std=train_std)
    rolling = rolling_drift_report(shuffled, window_size=WINDOW_SIZE, step_size=STEP_SIZE, min_window=MIN_WINDOW, train_mean=train_mean, train_std=train_std)
    sample_id = f'block_shuffle_{rep:03d}'
    null_rows.append({'source_type': 'null', 'sample_id': sample_id, 'rep': rep, 'n_obs': pooled['n_obs'], 'delta': pooled['delta'], 'pooled_delta': pooled['delta'], 'rolling_delta_mean': rolling['rolling_delta_mean'], 'rolling_delta_std': rolling['rolling_delta_std']})
    null_stylized_rows.append({'source_type': 'null', 'sample_id': sample_id, **stylized_report(shuffled, max_lag=MAX_LAG)})

if null_rows:
    null_df_tmp = pd.DataFrame(null_rows)
    null_stylized_tmp = pd.DataFrame(null_stylized_rows)
    drift_rows.append({
        'source_type': 'null',
        'sample_id': 'block_shuffle_summary',
        'n_obs': int(real_eval.size),
        'pooled_delta': float(null_df_tmp['delta'].mean()),
        'rolling_delta_mean': float(null_df_tmp['rolling_delta_mean'].mean()),
        'rolling_delta_std': float(null_df_tmp['rolling_delta_mean'].std(ddof=0)),
        'rolling_delta_min': float(null_df_tmp['delta'].min()),
        'rolling_delta_max': float(null_df_tmp['delta'].max()),
        'n_windows': int(NULL_REPS),
        'train_mean': train_mean,
        'train_std': train_std,
    })
    stylized_rows.append({
        'source_type': 'null',
        'sample_id': 'block_shuffle_summary',
        **{col: float(null_stylized_tmp[col].mean()) for col in null_stylized_tmp.select_dtypes(include=['number']).columns if col != 'n_obs'},
        'n_obs': int(real_eval.size),
    })

drift_df = pd.DataFrame(drift_rows)
component_df = pd.DataFrame(component_rows)
stylized_df = pd.DataFrame(stylized_rows)
null_df = pd.DataFrame(null_rows)

drift_df.to_csv(OUTPUT_DIR / 'drift_summary.csv', index=False)
component_df.to_csv(OUTPUT_DIR / 'drift_components.csv', index=False)
stylized_df.to_csv(OUTPUT_DIR / 'stylized_summary.csv', index=False)
null_df.to_csv(OUTPUT_DIR / 'null_summary.csv', index=False)

metadata = {
    'real_csv': str(REAL_CSV),
    'synthetic_csv': [str(path) for path in SYNTHETIC_CSVS],
    'input_type': INPUT_TYPE,
    'synthetic_input_type': SYNTHETIC_INPUT_TYPE,
    'value_column': VALUE_COLUMN,
    'date_column': DATE_COLUMN,
    'train_fraction': TRAIN_FRACTION,
    'val_fraction': VAL_FRACTION,
    'test_fraction': TEST_FRACTION,
    'eval_split': EVAL_SPLIT,
    'train_mean': train_mean,
    'train_std': train_std,
    'n_real_total': int(real_returns.size),
    'n_train': int(train.size),
    'n_eval': int(real_eval.size),
    'null_reps': NULL_REPS,
    'interpretation': 'Lower delta means less predictable drift under the fixed feature map; this is not a full martingale/no-arbitrage proof.',
}
(OUTPUT_DIR / 'metadata.json').write_text(json.dumps(metadata, indent=2, sort_keys=True))

report_table = drift_df.merge(stylized_df[['source_type', 'sample_id', 'acf_abs_r_lag1', 'excess_kurtosis']], on=['source_type', 'sample_id'], how='left')
report_table = report_table[['source_type', 'sample_id', 'n_obs', 'pooled_delta', 'rolling_delta_mean', 'rolling_delta_std', 'acf_abs_r_lag1', 'excess_kurtosis']]
report_md = '# Predictable-Drift Diagnostic Report\n\n' + report_table.to_markdown(index=False, floatfmt='.6g') + '\n\nLower delta means less predictable drift under the fixed feature map; this is not a full martingale/no-arbitrage proof.\n'
(OUTPUT_DIR / 'report.md').write_text(report_md)

print('Wrote outputs to:', OUTPUT_DIR.resolve())
display(report_table)

## 5. Download Outputs

In Colab, run the next cell to zip the output directory and download it.

In [ ]:
import shutil

zip_path = shutil.make_archive(str(OUTPUT_DIR), 'zip', OUTPUT_DIR)
print('Created:', zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as exc:
    print('Download helper is only available in Colab:', exc)

## Optional: Penalty Formula for Training Integration

If someone integrates this into a training loop with correctly ordered generated returns, the conservative extension is:

$$\mathcal{L}_{new}=\mathcal{L}_{FTS}+\lambda\widehat{\delta}^2,$$

$$\widehat{\delta}^2=\left\|\frac{1}{T}\sum_t r_{t+1}(1,r_t,r_{t-1},|r_t|,|r_{t-1}|)\right\|_2^2.$$

Choose `lambda` on validation only, then apply it unchanged to a final hold-out. Do not compute this across randomly shuffled mini-batches and call it the true diagnostic.